# Data Transformation

## 1. Merge all Clean Tables into a Temporary Table

In [0]:
%sql

USE CATALOG etl_project;
USE SCHEMA etl_schema;

CREATE OR REPLACE TEMP VIEW df_combined AS
SELECT * FROM df_2018
UNION ALL
SELECT * FROM df_2019
UNION ALL
SELECT * FROM df_2020
UNION ALL
SELECT * FROM df_2021;

## 2. Create New Temporary Table with Relevant Columns for Data Analysis 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_analysis_table AS
SELECT
    creation_date,
    YEAR(creation_date) AS year,
    MONTH(creation_date) AS month,
    entity_id,
    motivo,
    impacto,
    restricao_circulacao,
    morada,
    entity_id_flag
FROM df_combined;

## 3. Create Data Analysis Table 

In [0]:
%sql
USE CATALOG etl_project;
USE SCHEMA etl_schema;

CREATE OR REPLACE TABLE df_analysis_table AS
SELECT *
FROM df_analysis_table;

## 4. Create a Raw Combined Table for Comparison Analysis

In [0]:
%sql
USE CATALOG etl_project;
USE SCHEMA etl_schema;

CREATE OR REPLACE TABLE df_raw_combined AS

SELECT *
FROM (
    SELECT
        *,
        try_cast(creation_date AS TIMESTAMP) AS creation_date_clean
    FROM df_raw_2018

    UNION ALL

    SELECT
        *,
        try_cast(creation_date AS TIMESTAMP) AS creation_date_clean
    FROM df_raw_2019

    UNION ALL

    SELECT
        *,
        try_cast(creation_date AS TIMESTAMP) AS creation_date_clean
    FROM df_raw_2020

    UNION ALL

    SELECT
        *,
        try_cast(creation_date AS TIMESTAMP) AS creation_date_clean
    FROM df_raw_2021
) 
WHERE creation_date_clean IS NOT NULL;

In [0]:
%sql
ALTER TABLE etl_project.etl_schema.df_raw_combined
ADD COLUMN entity_id_flag INT;

In [0]:
%sql
UPDATE etl_project.etl_schema.df_raw_combined
SET entity_id_flag =
    CASE
        WHEN entity_id RLIKE '^EMEL\\.condicionamentoTransito\\.COND-2018-\\d+-\\d+$'
          OR entity_id RLIKE '^EMEL\\.condicionamentoTransito\\.COND-2019-\\d+-\\d+$'
          OR entity_id RLIKE '^EMEL\\.condicionamentoTransito\\.COND-2020-\\d+-\\d+$'
          OR entity_id RLIKE '^EMEL\\.condicionamentoTransito\\.COND-2021-\\d+-\\d+$'
        THEN 0
        ELSE 1
    END;